In [ ]:
from google import genai

client = genai.Client(api_key="")

sample_texts = [
    "다시는 보고 싶지 않은 짜증나는 영화",
    "아주 재미있는 영화",
    "정말 재미없는 영화였다",
    "이 영화 최고",
    "보통 영화"
]

for text in sample_texts:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"""다음 텍스트의 감성을 분석하세요.
반드시 '긍정', '부정', '중립' 중 하나만 답하세요. 다른 말은 하지 마세요.

텍스트: {text}"""
    )
    label = response.text.strip()
    print(f"{text} → {label}")

다시는 보고 싶지 않은 짜증나는 영화 → 부정
아주 재미있는 영화 → 긍정
정말 재미없는 영화였다 → 부정
이 영화 최고 → 긍정
보통 영화 → 중립


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델 로드 (이미 로드했으면 생략)
model_name = "naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-0.5B"
clovax_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
clovax_tokenizer = AutoTokenizer.from_pretrained(model_name)

sample_texts = [
    "다시는 보고 싶지 않은 짜증나는 영화",
    "아주 재미있는 영화",
    "정말 재미없는 영화였다",
    "이 영화 최고",
    "보통 영화"
]

for text in sample_texts:
    chat = [
        {"role": "tool_list", "content": ""},
        {"role": "system", "content": "당신은 감성 분석 전문가입니다."},
        {"role": "user", "content": f"""다음 텍스트의 감성을 분석하세요.
반드시 '긍정', '부정', '중립' 중 하나만 답하세요. 다른 말은 하지 마세요.

텍스트: {text}"""},
    ]

    inputs = clovax_tokenizer.apply_chat_template(
        chat,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(clovax_model.device)

    output_ids = clovax_model.generate(
        **inputs,
        max_new_tokens=10,
        repetition_penalty=1.2,
        eos_token_id=clovax_tokenizer.eos_token_id,
    )

    # 입력 부분 제외하고 생성된 토큰만 디코딩
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    result = clovax_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    print(f"{text} → {result}")

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

다시는 보고 싶지 않은 짜증나는 영화 → 감성 분석을 통해 이 문장의 감정을 파악해
